# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima8211/ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
### My lane as an ML task: Ranking

I will frame this lane as a ranking problem. The goal is to rank content pages by their priority for SEO or content review, rather than simply predicting whether a page should be refreshed or not. A ranking approach can help the content team focus first on pages that appear more likely to benefit from review based on multiple search-performance signals. The ranking would be used for decision-support, not as an automatic instruction to refresh a page.



In [1]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

url = "https://raw.githubusercontent.com/Fatima8211/ML_Internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nAvailable columns:")
print(df.columns.tolist())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows: 30000
Columns: 44

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target or proxy

My target will be a defined proxy for declining search performance. I will label a page as declining when its impressions in the last 30 days are less than 80% of its impressions in the previous 30 days. This label comes from an observed change in the starter data, but it is a rule-defined proxy rather than a directly observed business outcome such as a successful content refresh. The proxy will help me identify pages that may deserve higher priority for review.

In [2]:
# This cell is for CODE (numbers, a query, a check).
df["declining_proxy"] = (
    df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]
).astype(int)

print("Declining pages:", df["declining_proxy"].sum())
print("Total pages:", len(df))
print("Declining rate:", round(df["declining_proxy"].mean(), 3))

df[[
    "content_id",
    "impressions_last_30d",
    "impressions_prev_30d",
    "declining_proxy"
]].head(10)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Declining pages: 16262
Total pages: 30000
Declining rate: 0.542


,content_id,impressions_last_30d,impressions_prev_30d,declining_proxy
0,content_304f48230142,578,987,1
1,content_a1fb4e703a9e,2501,5915,1
2,content_9aa793d4d895,2382,6089,1
3,content_331d6c4de07b,3626,4206,0
4,content_d99b7a2d90ca,4211,6452,1
5,content_d4084a4bc775,617,1009,1
6,content_9a34b442b552,1,13,1
7,content_a63219c6e95a,636,632,0
8,content_5e6c160719bc,5696,13828,1
9,content_c27558df2b0c,252,356,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*
Success metric

I will use Precision@20 as the main success metric. It measures how many of the top 20 pages ranked as highest priority are actually labeled as declining by the defined proxy. A higher Precision@20 means the ranking is better at putting potentially declining pages near the top of the review list. I chose this metric because the content team may have limited time and may only be able to review a small number of pages first. This metric evaluates the usefulness of the top-priority recommendations rather than overall accuracy.

In [3]:
# This cell is for CODE (numbers, a query, a check).
def precision_at_k(scores, labels, k=20):
    top_k = pd.Series(scores).nlargest(k).index
    return labels.iloc[top_k].mean()

# Simple baseline: rank pages by their previous 30-day impressions.
baseline_scores = df["impressions_prev_30d"]

precision_20 = precision_at_k(
    baseline_scores,
    df["declining_proxy"],
    k=20
)

print(f"Baseline Precision@20: {precision_20:.3f}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Baseline Precision@20: 0.500


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis

The unit of analysis is one content page. Each row represents one anonymized content page identified by content_id. The features describe that page's search performance, content characteristics, freshness, and recent performance change. The target proxy is also defined at the page level, so each page receives its own declining or non-declining label.

In [4]:
# This cell is for CODE (numbers, a query, a check).
page_slice = df[[
    "content_id",
    "content_type",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "declining_proxy"
]].copy()

print("Rows:", len(page_slice))
print("Unique content pages:", page_slice["content_id"].nunique())

print("\nOne row represents one content page.")
print("\nSample of the page-level dataframe:")

page_slice.head(10)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows: 30000
Unique content pages: 30000

One row represents one content page.

Sample of the page-level dataframe:


,content_id,content_type,word_count,impressions_90d,clicks_90d,impressions_last_30d,impressions_prev_30d,content_age_days,days_since_last_update,ctr,avg_position,declining_proxy
0,content_304f48230142,keyword article,3221.0,3803,29,578,987,187,20,0.76,10.6,1
1,content_a1fb4e703a9e,keyword article,2481.0,15320,7,2501,5915,445,25,0.05,20.3,1
2,content_9aa793d4d895,keyword article,3515.0,12581,11,2382,6089,141,20,0.09,36.5,1
3,content_331d6c4de07b,keyword article,NaN,11751,58,3626,4206,463,22,0.49,6.2,0
4,content_d99b7a2d90ca,keyword article,2803.0,19140,24,4211,6452,263,14,0.13,44.0,1
5,content_d4084a4bc775,keyword article,3080.0,3970,1,617,1009,147,20,0.03,8.5,1
6,content_9a34b442b552,keyword article,3059.0,20,0,1,13,90,20,0.00,7.0,1
7,content_a63219c6e95a,keyword article,NaN,1724,1,636,632,445,22,0.06,21.2,0
8,content_5e6c160719bc,keyword article,3807.0,32574,29,5696,13828,90,20,0.09,46.0,1
9,content_c27558df2b0c,keyword article,NaN,1240,2,252,356,257,104,0.16,4.9,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
Why ML beats a fixed rule here

A fixed rule can be useful as a simple baseline, but it may be too limited for this problem because pages can have different combinations of search performance, content age, freshness, CTR, position, and other characteristics. For example, a 20% impression decline may mean something different for a new page than for an older page with a long history of performance. ML could combine several signals and learn patterns associated with the declining proxy instead of relying on one manually chosen threshold. The purpose would be to improve the prioritization of pages for human review, while still comparing the ML approach against a simple rule-based baseline.

In [5]:
# This cell is for CODE (numbers, a query, a check).
signals = [
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "search_volume"
]

print("Signals that could be used for ML ranking:")
for signal in signals:
    print("-", signal)

print("\nNumber of candidate signals:", len(signals))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Signals that could be used for ML ranking:
- impressions_prev_30d
- content_age_days
- days_since_last_update
- ctr
- avg_position
- search_volume

Number of candidate signals: 6


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.